# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides users in loading, exploring, and performing basic analysis on the FAIR^2 dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided as a Croissant schema JSON-LD file:
*URL*: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is available
!pip install mlcroissant

## 1. Data Loading
We use `mlcroissant` to load dataset metadata and records for inspection and processing.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access main metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")

## 2. Data Overview
Explore record sets, fields, and columns available in the dataset. All entities are referenced by their `@id`.

The Croissant schema can contain several record sets, each describing a logical collection of tabular records (e.g., the root protein table).

Let's list all available record sets and examine their structure.

In [ ]:
# Get all record sets by their @id
record_sets = list(dataset.record_sets.keys())
print(f"Found {len(record_sets)} record sets in the dataset:\n")
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"RecordSet @id: {rs_id}")
    print(f"  Name: {getattr(rs, 'name', None)}")
    print(f"  Description: {getattr(rs, 'description', None)}\n")
    print("  Fields:")
    for field_id in rs.fields:
        field = dataset.fields[field_id]
        print(f"    Field @id: {field_id}, name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction

Extract data from a chosen record set into a pandas DataFrame for analysis. **Replace `<Protein_RecordSet_ID>` with the target record set `@id` found above (likely representing proteins or main table).**

In [ ]:
# Example: Set this to the main record set relating to protein measurements (adjust accordingly)
# If you listed a RecordSet such as: 'https://api.app.sen.science/frontiers/7154140/PROT-RECORD-SET-ID'
main_record_set_id = list(dataset.record_sets.keys())[0]  # Use the first one if only one exists
print(f"Loading records from record set: {main_record_set_id}\n")

records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print("Columns available in DataFrame:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's perform some simple exploratory steps, referencing all fields by their `@id` as per best practice. Suppose you wish to analyze abundance—replace `<abundance_field_id>` and `<group_field_id>` with IDs found in the overview.

In [ ]:
# Find candidate numeric fields
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print("Numeric fields available:", numeric_candidates)

# For this analysis, select one numeric field, e.g., 'cr:abundance' (insert the actual @id)
numeric_field_id = numeric_candidates[0] if numeric_candidates else None

if numeric_field_id:
    # Filter records with value above threshold
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'fiu' else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field
    categorical_candidates = df.select_dtypes(include=["object"]).columns.tolist()
    print("Categorical fields available (candidate for grouping):", categorical_candidates)
    group_field_id = categorical_candidates[0] if categorical_candidates else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print('No numeric field detected for analysis.')

## 5. Visualization

Visualize the distribution of a numeric field from the chosen record set. Replace field names with their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot a histogram for the chosen numeric field
if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

## 6. Conclusion

* This notebook demonstrated programmatic access to a Croissant FAIR^2 dataset using `mlcroissant`.
* We explored available record sets, extracted core tabular data, and performed sample numeric EDA including filtering, normalization, and aggregation.
* For production usage, be sure to inspect the actual field `@id`s using the overview cells and adapt the notebook to your analytic goals and schema details.